# 文本分类实例

## Step1 导入相关包

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## Step2 加载数据

In [2]:
import pandas as pd

# 数据集是 https://github.com/SophonPlus/ChineseNlpCorpus
data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [3]:
# 去掉空行
data = data.dropna()
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


## Step3 创建 Dataset

In [4]:
from torch.utils.data import Dataset


class MyDataset(Dataset):

    def __init__(self) -> None:
        super().__init__()
        self.data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
        self.data = self.data.dropna()

    def __getitem__(self, index):
        return self.data.iloc[index]["review"], self.data.iloc[index]["label"]

    def __len__(self):
        return len(self.data)

In [5]:
dataset = MyDataset()
for i in range(5):
    print(dataset[i])

('距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.', np.int64(1))
('商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!', np.int64(1))
('早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。', np.int64(1))
('宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小，但加上低价位因素，还是无超所值的；环境不错，就在小胡同内，安静整洁，暖气好足-_-||。。。呵还有一大优势就是从宾馆出发，步行不到十分钟就可以到梅兰芳故居等等，京味小胡同，北海距离好近呢。总之，不错。推荐给节约消费的自助游朋友~比较划算，附近特色小吃很多~', np.int64(1))
('CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风', np.int64(1))


## Step4 划分数据集

In [6]:
from torch.utils.data import random_split


trainset, validset = random_split(dataset, lengths=[0.9, 0.1])
len(trainset), len(validset)

(6989, 776)

In [7]:
for i in range(10):
    print(trainset[i])

('对于酒店的服务态度基本满意,唯一缺陷就是没有遵守时间.我提前1小时和酒店总台确认过了我到达浦东机场的时间,并且酒店说我到的那个时间有班车在等.但在我比通知酒店的时间提前15分钟到达时,却没有酒店的班车,给总台打电话后才知道,班车已经提前开走了!结果酒店又排车来接,让我等待了25分钟!关于第二天早上出发时间的问题,我反复催促了,也是在晚上9点半以后才通知的.结果第二天的班车定在早上7点钟,但实际发车却延迟了15分钟!至少酒店的班车是没有信誉的!这样的工作态度好像很不在乎,像是在开玩笑一样!另外,酒店距离机场有至少25分钟的车程,周围也没有什么吃饭的地方.我到的当天晚饭只好在酒店餐厅解决.我只有1个人,点了个雪菜肉丝面,1个青茄子炒粉丝,结果上来的面至少够4个人吃,点菜时服务员也不提醒一下.那个青茄子炒粉丝就更甭提了,实在是难吃至极无法下咽!不知道是哪个菜有问题,当夜就上吐下泻了!可以肯定是食物中毒!酒店的小姐反复打电话到房间拉生意!实在是更加降低了酒店档次!又不敢把电话线拔掉,因为还要等待总台通知第二天早上发车时间,真是太郁闷了!188元实在是太便宜了,便宜得很难受很不合算,我永远也不会再去住这家酒店了!', np.int64(0))
('我在尼斯住了5天，价格不是很贵，服务比较细腻，早餐也比较丰富，感觉还是不错。', np.int64(1))
('一家欧式风格的酒店，大堂气派，房间也不错，还有水果、咖啡送。“美林阁”餐厅在上海挺有名气，特地去品尝了一下，的确不错，凭房卡还可以打8.8折。一楼大堂西侧有一个酒吧，晚上和几个朋友一起小坐了一会儿，里面环境不错，东西也不贵，同样凭房卡还可以打8.8折。下次还会入住这家酒店，有兴趣的朋友不妨也去试试。', np.int64(1))
('房间很大，而且直接面对长江，景色很漂亮，不过附近好象吃饭地方少了点。', np.int64(1))
('前台小姐的态度不好，房间一开门走进去，一股发霉的味道，而且阳台的门的玻璃是坏的，cheakout的时候前台还让我们赔钱，说我们把门弄坏了。前台小姐兴冲冲的跑上来说隔壁的客人投诉我们太吵，要我们做房客登记，态度差的一塌糊涂，隔壁是客人，难道我们不是客人吗？而且我们一开门，隔壁比我们吵多了。反正就是一个字：差！', np.int64(0))
('酒店位置不错房间一般但是早餐还是不错的餐厅比较

In [8]:
for i in range(10):
    print(validset[i])

('环境幽雅，房间卫生，空气清新，服务好，安全！！饮食比较好，价位公道，比那些路边店好的多！！', np.int64(1))
('我于4月底入住该酒店，整体来说该酒店超出我的预期。酒店门口有公交车站，多路公交车经过；还有出租车在门口，打车也很方便。酒店客房层设有中庭，宽敞明亮，让人感觉眼前一亮。标房面积很大，还配有电脑提供免费上网。服务员非常热情，能主动问候并提供建议。某种程度上感觉是五星级酒店，下次还会来。', np.int64(1))
('酒店的整体服务意识相当好，从办理登记入住、用餐、退房等方面都是五星级的服务。入住当晚我们想外出品尝当地特色菜，随意咨询到酒店前台员工，他们主动给我们介绍，最后还利用下班时间陪同我们外出，给我们特别深刻的印象，以后出差，我还会选择富盈酒店。', np.int64(1))
('门口服务很好,一楼的湖南菜价格也不贵,味道很好!尤其是菜单上没有的菜,他们会安排二楼的餐厅去做.一个人吃饭的话，汤可以按碗来点,这个很爽!有8元/10元/碗的汤,很好喝.早餐简陋了点，我喜欢喝酸奶,可这早上不提供.房间也是挺干净的.我当时是去电力机车厂办事,很近!只是叫的有时候真的不很', np.int64(1))
('房间设施很好，服务很周到．特别是门童和前台，很热情．酒店每晚都征求意见，看有什么不满意的，这是其他酒店没有的．', np.int64(1))
('房间设施太旧、太差，环境也不好，房间大而无用。非常不喜欢', np.int64(0))
('广泛看了网上的点评，最后选择了这家酒店，3楼的商务海景标间，景色确实一流，就是隔音效果差了点，原来还准备选海天，实地一看才知道原来是竖着对海，房间都是侧看海。前台听说是协程的客人，有点冷淡。大眼睛的大堂经理确实好！', np.int64(1))
('这是我住酒店以来碰到最郁闷和龌龊的酒店,我是带着快8个月的儿子自己开车去的,亲戚已经开好其他宾馆,就是考虑儿子需要有个好环境,才决定来此酒店开一间大床房,入住的是我妈我媳妇我儿子,房间小了点但看着还温馨.开始都相对满意.进去待了一段时间就发现没空调的,儿子怕热闹的厉害,和酒店交涉了一下,结果给我们扛了把落地风扇...住4星扇风扇(就店对此事的2个说法,开始告诉我们说是因为气候没到开空调的时候,所以中央空调没打开,4星说出这话我很佩服,后来发生了更严重的事后,再提起

## Step5 创建 Dataloader

In [9]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")


def collate_func(batch):
    # print("batch:", batch)

    texts, labels = [], []
    for item in batch:
        texts.append(item[0])
        labels.append(item[1])
    # print("texts:", texts)
    # print("labels:", labels)

    inputs = tokenizer(
        texts,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    inputs["labels"] = torch.tensor(labels)

    return inputs


"""
batch: [
    ('房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！', np.int64(1)),
    ('酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。', np.int64(1)),
    ('入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚度不够5星的。家具大都是原色的，地毯也是颜色的，不亮丽。电视陈旧，遥控器不灵了。比较好玩的是房间里的电水壶的大个头，好像还是进口的。还有早餐分中西常规、粤式(我倾向于这个)和日式的3个厅，不过只能选择其中一个，到10点结束。走廊是分片的感应灯。有夜床服务、午茶小饼干1块、枕头上会放一张别着小鲜花的印有诗句的卡纸，赞！洁具质量不好，除了肥皂。无烟楼层有客人在走廊和房间(房门洞开)抽烟，无人制止。缺少报纸供应。盥洗室放有女孩子可应急的发筋，细心，再赞一个。套房的景色很不错，离海倒是确实近，就在背后。稍微偏离一点商业区，不过不远，打车到五四广场3公里不到。补充点评2008年3月6日：差点忘了，我打车的司机径直把握开到东楼，拉门的先生还给我一张记录有车牌号的纸片，赞一下。另外，青岛的酒店服务普遍不错。不过饭店的服务员可能没听说过“菊花普洱”。', np.int64(1))
]

texts: [
    '房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！',
    '酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。',
    '入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚度不够5星的。家具大都是原色的，地毯也是颜色的，不亮丽。电视陈旧，遥控器不灵了。比较好玩的是房间里的电水壶的大个头，好像还是进口的。还有早餐分中西常规、粤式(我倾向于这个)和日式的3个厅，不过只能选择其中一个，到10点结束。走廊是分片的感应灯。有夜床服务、午茶小饼干1块、枕头上会放一张别着小鲜花的印有诗句的卡纸，赞！洁具质量不好，除了肥皂。无烟楼层有客人在走廊和房间(房门洞开)抽烟，无人制止。缺少报纸供应。盥洗室放有女孩子可应急的发筋，细心，再赞一个。套房的景色很不错，离海倒是确实近，就在背后。稍微偏离一点商业区，不过不远，打车到五四广场3公里不到。补充点评2008年3月6日：差点忘了，我打车的司机径直把握开到东楼，拉门的先生还给我一张记录有车牌号的纸片，赞一下。另外，青岛的酒店服务普遍不错。不过饭店的服务员可能没听说过“菊花普洱”。'
]

labels: [np.int64(1), np.int64(1), np.int64(1)]
"""

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


"\nbatch: [\n    ('房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！', np.int64(1)),\n    ('酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。', np.int64(1)),\n    ('入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚度不够5星的。家具大都是原色的，地毯也是颜色的，不亮丽。电视陈旧，遥控器不灵了。比较好玩的是房间里的电水壶的大个头，好像还是进口的。还有早餐分中西常规、粤式(我倾向于这个)和日式的3个厅，不过只能选择其中一个，到10点结束。走廊是分片的感应灯。有夜床服务、午茶小饼干1块、枕头上会放一张别着小鲜花的印有诗句的卡纸，赞！洁具质量不好，除了肥皂。无烟楼层有客人在走廊和房间(房门洞开)抽烟，无人制止。缺少报纸供应。盥洗室放有女孩子可应急的发筋，细心，再赞一个。套房的景色很不错，离海倒是确实近，就在背后。稍微偏离一点商业区，不过不远，打车到五四广场3公里不到。补充点评2008年3月6日：差点忘了，我打车的司机径直把握开到东楼，拉门的先生还给我一张记录有车牌号的纸片，赞一下。另外，青岛的酒店服务普遍不错。不过饭店的服务员可能没听说过“菊花普洱”。', np.int64(1))\n]\n\ntexts: [\n    '房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！',\n    '酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。',\n    '入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚

In [10]:
# 调试
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=3, shuffle=False, collate_fn=collate_func)

In [11]:
next(enumerate(trainloader))

(0,
 {'input_ids': tensor([[ 101, 2190,  754, 6983, 2421, 4638, 3302, 1218, 2578, 2428, 1825, 3315,
          4007, 2692,  117, 1546,  671, 5375, 7379, 2218, 3221, 3766, 3300, 6905,
          2127, 3198, 7313,  119, 2769, 2990, 1184,  122, 2207, 3198, 1469, 6983,
          2421, 2600, 1378, 4802, 6371, 6814,  749, 2769, 1168, 6809, 3855,  691,
          3322, 1767, 4638, 3198, 7313,  117, 2400,  684, 6983, 2421, 6432, 2769,
          1168, 4638, 6929,  702, 3198, 7313, 3300, 4408, 6756, 1762, 5023,  119,
           852, 1762, 2769, 3683, 6858, 4761, 6983, 2421, 4638, 3198, 7313, 2990,
          1184, 8115, 1146, 7164, 1168, 6809, 3198,  117, 1316, 3766, 3300, 6983,
          2421, 4638, 4408, 6756,  117, 5314, 2600, 1378, 2802, 4510, 6413, 1400,
          2798, 4761, 6887,  117, 4408, 6756, 2347, 5307, 2990, 1184, 2458, 6624,
           749,  106, 5310, 3362, 6983, 2421, 1348,  102],
         [ 101, 2769, 1762, 2225, 3172,  857,  749,  126, 1921, 8024,  817, 3419,
           679, 3221,

In [12]:
len(trainloader)

2330

In [13]:
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=32, shuffle=True, collate_fn=collate_func)
validloader = DataLoader(
    validset, batch_size=64, shuffle=False, collate_fn=collate_func
)

## Step6 创建模型及优化器

In [14]:
from torch.optim import Adam

model = AutoModelForSequenceClassification.from_pretrained("hfl/rbt3")

if torch.cuda.is_available():
    model = model.cuda()

model.device

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


device(type='cuda', index=0)

In [15]:
optimizer = Adam(model.parameters(), lr=2e-5)

## Step7 训练与验证

In [16]:
def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output.logits, dim=-1)
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    return acc_num / len(validset)


def train(epoch=3, log_step=10):
    global_step = 0
    for ep in range(epoch):

        model.train()

        # 每个 epoch 训练全部数据
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}

            optimizer.zero_grad()
            output = model(**batch)
            output.loss.backward()
            optimizer.step()

            if global_step % log_step == 0:
                print(
                    f"ep: {ep}, global_step: {global_step}, loss: {output.loss.item()}"
                )

            global_step += 1

        acc = evaluate()
        print(f"ep: {ep}, acc: {acc}")

## Step8 模型训练

In [17]:
train()

ep: 0, global_step: 0, loss: 0.6166554689407349
ep: 0, global_step: 10, loss: 0.5847089290618896
ep: 0, global_step: 20, loss: 0.5405674576759338
ep: 0, global_step: 30, loss: 0.35793817043304443
ep: 0, global_step: 40, loss: 0.15935686230659485
ep: 0, global_step: 50, loss: 0.35054248571395874
ep: 0, global_step: 60, loss: 0.3162122964859009
ep: 0, global_step: 70, loss: 0.29032576084136963
ep: 0, global_step: 80, loss: 0.2839092016220093
ep: 0, global_step: 90, loss: 0.36454614996910095
ep: 0, global_step: 100, loss: 0.22261105477809906
ep: 0, global_step: 110, loss: 0.4918762147426605
ep: 0, global_step: 120, loss: 0.22107675671577454
ep: 0, global_step: 130, loss: 0.17775504291057587
ep: 0, global_step: 140, loss: 0.25741010904312134
ep: 0, global_step: 150, loss: 0.2849391996860504
ep: 0, global_step: 160, loss: 0.3004488945007324
ep: 0, global_step: 170, loss: 0.3044661581516266
ep: 0, global_step: 180, loss: 0.14435474574565887
ep: 0, global_step: 190, loss: 0.4201275706291199
e

## Step9 模型预测

In [18]:
sen = "我觉得这家酒店不错，饭很好吃！"
id2_label = {0: "差评！", 1: "好评！"}
model.eval()
with torch.inference_mode():
    inputs = tokenizer(sen, return_tensors="pt")
    inputs = {k: v.cuda() for k, v in inputs.items()}
    logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=-1)
    print(f"输入：{sen}\n模型预测结果:{id2_label.get(pred.item())}")

输入：我觉得这家酒店不错，饭很好吃！
模型预测结果:好评！


In [19]:
from transformers import pipeline

model.config.id2label = id2_label
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [20]:
pipe(sen)

[{'label': '好评！', 'score': 0.9915199875831604}]